In [ ]:
!pip install evaluate wandb emoji==0.6.0
seed = 42

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
my_secret = user_secrets.get_secret("WANDB_API_KEY") 
wandb.login(key=my_secret)

# Import data

In [ ]:
import pandas as pd

path_ds = "/kaggle/input/nlp-getting-started/"

df_train = pd.read_csv(path_ds+"train.csv")
df_test = pd.read_csv(path_ds+"test.csv")

# EDA

In [ ]:
len_df_train = len(df_train)
#print("How many tweets? ", len_df_train) #7613

# Does the column 'id' contain unique values? yes
#print('len(df_train["id"].unique()) == len(df_train)?', len(df_train["id"].unique()) == len_df_train)

# Let's see how 0s and 1s in the "id" column are distributed
target_g = df_train[['id', 'target']].groupby('target').count().rename(columns={"id": "count_id"})
target_g['percentage_id'] = target_g["count_id"]/len_df_train*100
display(target_g) # 57% of the tweets are not about real disasters

# How many words are in "text" for each row?
df_train['text_split'] = df_train['text'].str.split()
df_train['n_tokens'] = [len(e) for e in df_train['text_split']]
print(df_train['n_tokens'].unique())
df_train['n_tokens'].hist()

df_train[df_train['n_tokens']<=5]

# Data cleaning

In [ ]:
"""
# Probably links will not tell us anything useful... I also need to lowercase the text and deal with contractions

def remove_accents(df):
    # remove accents
    df['text'] = df['text'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    # this is not a accents but i need to deal with this
    df['text'] = df['text'].str.replace("&amp;", " and ")   
    return df

def lowercase(df):
    df['text'] = df['text'].str.lower()
    return df

def remove_contractions(df):
    # deal with contraction: don't, isn't, aren't, weren't, shouldn't, couldn't, won't (will not), ...
    df['text'] = df['text'].str.replace("won't", 'will not')
    df['text'] = df['text'].str.replace("n't", ' not')
    return df    

def adjust_text_data(df):
    # remove links
    df['text'] = [' '.join([e for e in row.split() if 'http' not in e]) for row in df['text']]

    df = remove_accents(df)
    df = lowercase(df)
    df = remove_contractions(df)

    if 'target' in df.columns:
        df = df.rename(columns = {'target': 'labels'})
    
    return df


# I would like to add keyword and location within the text column --> they might have useful info

def add_keyword_and_location(df):
    # adjust the keyword columns
    df['keyword'] = df['keyword'] .str.replace('%20', ' ')
    # put together the text columns aka 'keyword', 'location', 'text'
    for col in ['keyword', 'location', 'text']:
        df[col] = col + ': ' + df[col].astype(str)
    df['text'] = df['keyword'] + '; ' + df['location'] + '; ' + df['text']

    '''
    df = remove_accents(df)
    df = lowercase(df)
    df = remove_contractions(df)
    '''
    df = df.drop(columns = ['keyword', 'location'])
    return df

"""

In [ ]:
!git clone https://github.com/VinAIResearch/BERTweet.git
!ls

In [ ]:
import sys
sys.path.append('./BERTweet')

from TweetNormalizer import normalizeTweet

"""
# adjust data
df_train = adjust_text_data(df_train)
df_test = adjust_text_data(df_test)

# adding keyword and location to text
#print(df_train['keyword'].unique()) 
print(df_train[['keyword']].drop_duplicates().keyword.to_list()[:20]) # from here i see that %20 is used instead of spaces
print(df_train[['location']].drop_duplicates().location.to_list()[:20])
df_train = add_keyword_and_location(df_train)
df_test = add_keyword_and_location(df_test)
"""

df_train['text'] = df_train["text"].apply(normalizeTweet)
df_train = df_train.rename(columns = {'target': 'labels'})
df_test['text'] = df_test["text"].apply(normalizeTweet)

df_train = df_train[['id', 'text', 'labels']]
display(df_train[['id', 'text', 'labels']])

# Creating validation dataset and transforming DataFrame into HF Dataset

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
#from sklearn.model_selection import train_test_split

# I want a validation dataset

total_data = len(df_train)+len(df_test)
print("train", len(df_train)/total_data*100)
print("test", len(df_test)/total_data*100)

# Use StratifiedShuffleSplit to create train/validation indices --> this splitter balances classes in train and validation sets
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)  
for train_idx, valid_idx in splitter.split(df_train['text'], df_train['labels']):
    break  # just get the first split
df_valid = df_train.iloc[valid_idx, :].reset_index(drop=True)
df_train = df_train.iloc[train_idx, :].reset_index(drop=True)

display(df_train.head())
display(df_valid.head())

In [ ]:
from datasets import Dataset, DatasetDict

ds = DatasetDict(
    {
        "train": Dataset.from_pandas(df_train),
        "test": Dataset.from_pandas(df_test),
        "validation":  Dataset.from_pandas(df_valid)
    })

ds

# Pre processing

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

#checkpoint = "bert-base-uncased"
#checkpoint = "distilbert-base-uncased"
checkpoint = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenized_datasets = ds.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training and evaluation (for every epoch)

In [ ]:
import evaluate
import numpy as np

def compute_metrics(eval_preds):
    metric1 = evaluate.load("f1")
    metric2 = evaluate.load("accuracy")
    
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    f1 = metric1.compute(predictions=predictions, references=labels)["f1"]
    accuracy = metric2.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": accuracy}

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

In [ ]:
from transformers import TrainingArguments

# Initialize Weights & Biases for experiment tracking
wandb.init(project="transformer-fine-tuning", name="bert-nlp-analysis")

"""
training_args = TrainingArguments(
    "test-trainer", 
    eval_strategy="epoch")


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,  # Log metrics every 10 steps
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="wandb",  # Send logs to Weights & Biases
)
"""

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_strategy="steps",
    logging_steps=10,  # Log metrics every 10 steps
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    num_train_epochs=10,  # Set high, but we'll stop early
    
    weight_decay=0.01,  # regularization to reduce overfitting
    
    learning_rate=2e-5,
    lr_scheduler_type="linear",  # or cosine, cosine_with_restarts, polynomial, etc.
    warmup_steps=100,  # or warmup_ratio

    #label_smoothing_factor=0.1,
    
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="wandb",  # Send logs to Weights & Biases
)


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)], # stops if no improvement for 2 evals
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_datasets["test"])
#print(predictions.predictions.shape, predictions.label_ids.shape)
preds = np.argmax(predictions.predictions, axis=-1)
preds[:10]

In [ ]:
df_test["target"] = preds
display(df_test)
df_test = df_test[['id', 'target']]
df_test.to_csv('submission.csv', index = False)

# check of optimal threshold

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve
import math

# custom function
def sigmoid(x):
  return 1 / (1 + math.exp(-x))
# define vectorized sigmoid
sigmoid_v = np.vectorize(sigmoid)

# Get prediction probabilities on validation set
predictions_valid = trainer.predict(tokenized_datasets["validation"])
sig_pred = sigmoid_v(predictions_valid.predictions)
print(sig_pred[:2])
val_probs = np.max(sig_pred, axis = -1)
print(val_probs[:2])

# Calculate precision-recall curve
y_val = df_valid['labels'].to_list()
precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
average_precision = average_precision_score(y_val, val_probs)

# Find optimal threshold based on F1 score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)  # Add small epsilon to avoid division by zero
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal threshold: {optimal_threshold:.3f}")
print(f"F1 score at optimal threshold: {f1_scores[optimal_idx]:.3f}")